# Dana’s Bakery — deterministic LangGraph orders
Upload this notebook to Google Colab and choose **Runtime → Run all** in a standard CPU runtime. No credentials are needed. Each order uses the same fixed inventory; nothing is reserved or persisted. Ordinary Python nodes validate, select at most two alternatives, choose fulfilment, calculate payment, and write a friendly reply.

## 1. Install dependencies
Internet is needed to install LangGraph and render the diagram using its default Mermaid rendering service. Order processing uses no network calls.

In [ ]:
%pip install -q -U langgraph


## 2. Imports

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from pprint import pprint

from langgraph.graph import StateGraph, START, END


## 3. Business constants
Rugelach quantities count boxes. The final finance rule is 15% off goods above 150 NIS except Friday; delivery is never discounted.

In [ ]:
MENU = {"challah": 18.0, "babka": 45.0, "rugelach": 30.0, "sourdough": 25.0}
INVENTORY = {"challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4}
NEXT_ITEM = {"babka": "rugelach", "rugelach": "sourdough", "sourdough": "challah", "challah": "babka"}
DAYS = {day.lower(): day for day in (
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
)}
REQUIRED_INPUTS = ("item", "quantity", "day", "city")
MAX_SUBSTITUTES = 2
DELIVERY_FEE = 20.0
DISCOUNT_THRESHOLD = 150.0
DISCOUNT_RATE = 0.15
NO_DISCOUNT_DAY = "Friday"

## 4. Shared state
Input fields accept `object` because validation must handle incorrect types explicitly. `total=False` permits missing keys to reach validation. An input schema exposes only the four customer inputs; all derived fields are reset by initialization.

In [ ]:
class OrderInput(TypedDict, total=False):
    item: object
    quantity: object
    day: object
    city: object


class OrderState(OrderInput, total=False):
    status: Literal["accepted", "rejected"]
    delivery: Literal["pickup", "delivery"]
    total: float
    message: str
    steps: list[str]
    requested_item: str
    current_item: str
    missing_inputs: list[str]
    validation_errors: list[str]
    valid: bool
    in_stock: bool
    suggestion_count: int
    unit_price: float
    goods_subtotal: float
    discounted_goods: float
    discount_amount: float
    delivery_fee: float
    outside_jerusalem: bool
    substituted: bool

## 5. Nodes and routing
Each node returns updates with a new diary list. Routers select edges without adding fictional node visits. The second alternative always passes through `check_stock` before cancellation is possible.

In [ ]:
def initialize_order(state: OrderState) -> dict:
    raw_item = state.get("item")
    requested = raw_item.strip().lower() if isinstance(raw_item, str) else ""
    raw_day = state.get("day")
    day = raw_day.strip() if isinstance(raw_day, str) else raw_day
    if isinstance(day, str):
        day = DAYS.get(day.lower(), day)
    raw_city = state.get("city")
    city = "" if raw_city is None else raw_city
    if isinstance(city, str):
        city = city.strip()
        if city.lower() == "jerusalem":
            city = "Jerusalem"
    return {
        "item": requested, "quantity": state.get("quantity"), "day": day, "city": city,
        "requested_item": requested, "current_item": requested,
        "missing_inputs": [key for key in REQUIRED_INPUTS if key not in state],
        # Keep raw-item type validation separate from the normalized output item.
        "validation_errors": [] if isinstance(raw_item, str) else ["item must be a string"],
        "valid": False, "in_stock": False, "suggestion_count": 0,
        "status": "rejected", "delivery": "pickup", "total": 0.0, "message": "",
        "unit_price": 0.0, "goods_subtotal": 0.0, "discounted_goods": 0.0,
        "discount_amount": 0.0, "delivery_fee": 0.0,
        "outside_jerusalem": False, "substituted": False,
        "steps": [f"initialize_order: requested {requested!r}; fresh order state."],
    }


def validate_order(state: OrderState) -> dict:
    errors = list(state["validation_errors"])
    if state["missing_inputs"]:
        errors.append("missing required fields: " + ", ".join(state["missing_inputs"]))
    if state["requested_item"] not in MENU:
        errors.append("please choose a product from the current menu")
    if type(state["quantity"]) is not int or state["quantity"] <= 0:
        errors.append("quantity must be a positive integer")
    if not isinstance(state["day"], str) or state["day"] not in DAYS.values():
        errors.append("day must be a valid weekday name")
    if not isinstance(state["city"], str):
        errors.append("city must be a string or None")
    decision = "failed: " + "; ".join(errors) if errors else "passed"
    return {"valid": not errors, "validation_errors": errors,
            "steps": state["steps"] + [f"validate_order: {decision}."]}


def route_validation(state: OrderState) -> str:
    return "valid" if state["valid"] else "invalid"


def reject_invalid_order(state: OrderState) -> dict:
    return {"status": "rejected", "total": 0.0, "delivery": "pickup",
            "message": "Thank you for thinking of Dana's Bakery! We couldn't process your order: "
                       + "; ".join(state["validation_errors"]) + ". Please try again; we'd love to help.",
            "steps": state["steps"] + ["reject_invalid_order: rejected; no stock or pricing checks; total 0.00 NIS."]}


def check_stock(state: OrderState) -> dict:
    candidate = state["current_item"]
    enough = INVENTORY[candidate] >= state["quantity"]
    label = "requested product" if state["suggestion_count"] == 0 else f"substitute {state['suggestion_count']}"
    return {"in_stock": enough,
            "steps": state["steps"] + [
                f"check_stock: {label} {candidate}; need {state['quantity']}, "
                f"available {INVENTORY[candidate]}; " + ("sufficient." if enough else "insufficient.")
            ]}


def route_stock(state: OrderState) -> str:
    if state["in_stock"]:
        return "available"
    return "substitute" if state["suggestion_count"] < MAX_SUBSTITUTES else "cancel"


def try_substitute(state: OrderState) -> dict:
    candidate = NEXT_ITEM[state["current_item"]]
    count = state["suggestion_count"] + 1
    return {"current_item": candidate, "suggestion_count": count,
            "steps": state["steps"] + [
                f"try_substitute: selected alternative {count}/{MAX_SUBSTITUTES}: {candidate}; quantity unchanged."
            ]}


def cancel_unavailable_order(state: OrderState) -> dict:
    return {"status": "rejected", "item": state["requested_item"],
            "total": 0.0, "delivery": "pickup",
            "message": f"We're sorry! We cannot supply {state['quantity']} of {state['requested_item']} "
                       "or either of the two alternatives. Your order is cancelled with no charge. "
                       "We hope to bake for you another day!",
            "steps": state["steps"] + [
                "cancel_unavailable_order: requested quantity unavailable after two alternatives; rejected; total 0.00 NIS."
            ]}


def determine_delivery(state: OrderState) -> dict:
    delivery = "delivery" if state["city"] == "Jerusalem" else "pickup"
    outside = bool(state["city"]) and delivery == "pickup"
    decision = "Jerusalem delivery, fee 20.00 NIS" if delivery == "delivery" else (
        "outside Jerusalem; changed to free pickup" if outside else "ordinary free pickup"
    )
    sold_item = state["current_item"]
    return {"status": "accepted", "item": sold_item, "unit_price": MENU[sold_item],
            "substituted": sold_item != state["requested_item"],
            "delivery": delivery, "outside_jerusalem": outside,
            "delivery_fee": DELIVERY_FEE if delivery == "delivery" else 0.0,
            "steps": state["steps"] + [f"determine_delivery: {decision}."]}


def calculate_total(state: OrderState) -> dict:
    subtotal = round(state["unit_price"] * state["quantity"], 2)
    eligible = subtotal > DISCOUNT_THRESHOLD and state["day"] != NO_DISCOUNT_DAY
    discounted = round(subtotal * (1 - DISCOUNT_RATE) if eligible else subtotal, 2)
    total = round(discounted + state["delivery_fee"], 2)
    reason = "15% goods discount applied" if eligible else (
        "discount skipped: Friday" if state["day"] == NO_DISCOUNT_DAY
        else "discount skipped: goods subtotal is not above 150 NIS"
    )
    return {"goods_subtotal": subtotal, "discounted_goods": discounted,
            "discount_amount": round(subtotal - discounted, 2), "total": total,
            "steps": state["steps"] + [
                f"calculate_total: {reason}; goods {subtotal:.2f} -> {discounted:.2f}; "
                f"delivery {state['delivery_fee']:.2f}; final total {total:.2f} NIS."
            ]}


def create_message(state: OrderState) -> dict:
    parts = ["Thank you for ordering from Dana's Bakery!"]
    if state["substituted"]:
        parts.append(f"{state['requested_item'].capitalize()} is unavailable in your quantity, "
                     f"so we've substituted {state['item']} at its own price.")
    unit = "box(es) of rugelach" if state["item"] == "rugelach" else state["item"]
    parts.append(f"Your order for {state['quantity']} {unit} is accepted.")
    if state["delivery"] == "delivery":
        parts.append("We'll deliver inside Jerusalem for 20.00 NIS.")
    elif state["outside_jerusalem"]:
        parts.append("Delivery is available only inside Jerusalem, so your order is ready for free pickup at the bakery.")
    else:
        parts.append("We look forward to seeing you for free pickup at the bakery!")
    if state["discount_amount"]:
        parts.append(f"Your 15% baked-goods discount saves {state['discount_amount']:.2f} NIS.")
    parts.append(f"Your total is {state['total']:.2f} NIS. Enjoy!")
    return {"message": " ".join(parts), "steps": state["steps"] + [
        f"create_message: accepted reply prepared; total {state['total']:.2f} NIS."
    ]}

## 6. Construct and compile the graph
The stock router runs for the requested product and for every selected substitute. Its three explicit branches appear in the diagram. No checkpointer or persistent store is used.

In [ ]:
builder = StateGraph(OrderState, input_schema=OrderInput)
for node in (initialize_order, validate_order, reject_invalid_order, check_stock,
             try_substitute, cancel_unavailable_order, determine_delivery,
             calculate_total, create_message):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "initialize_order")
builder.add_edge("initialize_order", "validate_order")
builder.add_conditional_edges("validate_order", route_validation, {
    "valid": "check_stock", "invalid": "reject_invalid_order"
})
builder.add_conditional_edges("check_stock", route_stock, {
    "available": "determine_delivery", "substitute": "try_substitute",
    "cancel": "cancel_unavailable_order"
})
builder.add_edge("try_substitute", "check_stock")
builder.add_edge("reject_invalid_order", END)
builder.add_edge("cancel_unavailable_order", END)
builder.add_edge("determine_delivery", "calculate_total")
builder.add_edge("calculate_total", "create_message")
builder.add_edge("create_message", END)
graph = builder.compile()

## 7. Display the actual graph
Display the compiled graph using LangGraph’s default Mermaid PNG renderer. This cell requires internet access; no browser installation is needed.


In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))


## 8. Reusable order runner
Every printed result includes the six public fields and Avi’s chronological audit diary. Assertions check output types, rejection defaults, bounded substitution, and exactly which nodes executed.

In [ ]:
PUBLIC_KEYS = {"status", "item", "delivery", "total", "message", "steps"}


def run_order(label, order, expected):
    print(f"\n{'=' * 70}\n{label}\nInput:")
    pprint(order)
    result = graph.invoke(order)
    assert PUBLIC_KEYS <= result.keys()
    assert result["status"] in {"accepted", "rejected"}
    assert result["delivery"] in {"pickup", "delivery"}
    assert isinstance(result["message"], str) and result["message"]
    assert all(isinstance(step, str) for step in result["steps"])
    assert 0 <= result["suggestion_count"] <= MAX_SUBSTITUTES
    assert result["quantity"] == order.get("quantity")
    for key, value in expected.items():
        assert result[key] == value, (label, key, result[key], value)
    if result["status"] == "rejected":
        assert result["total"] == 0.0 and result["delivery"] == "pickup"
        assert result["item"] == result["requested_item"]
    visits = [step.split(":", 1)[0] for step in result["steps"]]
    if not result["valid"]:
        assert visits == ["initialize_order", "validate_order", "reject_invalid_order"]
    else:
        stock_visits = [step for step in result["steps"] if step.startswith("check_stock:")]
        assert len(stock_visits) == 1 + result["suggestion_count"]
        candidate = result["requested_item"]
        for index, step in enumerate(stock_visits):
            if index:
                candidate = NEXT_ITEM[candidate]
            assert f" {candidate};" in step
        expected_visits = ["initialize_order", "validate_order", "check_stock"]
        expected_visits += ["try_substitute", "check_stock"] * result["suggestion_count"]
        if result["status"] == "accepted":
            expected_visits += ["determine_delivery", "calculate_total", "create_message"]
            assert result["unit_price"] == MENU[result["item"]]
            assert result["goods_subtotal"] == MENU[result["item"]] * result["quantity"]
            assert result["total"] == round(result["discounted_goods"] + result["delivery_fee"], 2)
        else:
            expected_visits += ["cancel_unavailable_order"]
            assert result["suggestion_count"] == 2
        assert visits == expected_visits
    print("Result:")
    pprint({key: result[key] for key in ("status", "item", "delivery", "total", "message")})
    print("Avi's diary:")
    for number, step in enumerate(result["steps"], 1):
        print(f"  {number}. {step}")
    return result

## 9. The twelve required scenarios

In [ ]:
TEST_CASES = [
    ("1. Jerusalem, no discount", "challah", 2, "Thursday", "Jerusalem", "accepted", "challah", "delivery", 56.0),
    ("2. Empty city", "sourdough", 2, "Monday", "", "accepted", "sourdough", "pickup", 50.0),
    ("3. Outside Jerusalem", "rugelach", 2, "Tuesday", "Tel Aviv", "accepted", "rugelach", "pickup", 60.0),
    ("4. First substitute", "babka", 2, "Thursday", "Jerusalem", "accepted", "rugelach", "delivery", 80.0),
    ("5. Second substitute", "babka", 4, "Wednesday", "", "accepted", "sourdough", "pickup", 100.0),
    ("6. Two substitutes fail", "babka", 8, "Thursday", "Jerusalem", "rejected", "babka", "pickup", 0.0),
    ("7. Discount on goods only", "challah", 10, "Thursday", "Jerusalem", "accepted", "challah", "delivery", 173.0),
    ("8. Friday", "challah", 10, "Friday", "Jerusalem", "accepted", "challah", "delivery", 200.0),
    ("9. Circular order", "challah", 11, "Monday", "", "rejected", "challah", "pickup", 0.0),
    ("10. Invalid quantity", "challah", 0, "Monday", "Jerusalem", "rejected", "challah", "pickup", 0.0),
    ("11. Unsupported item", "pizza", 1, "Monday", "Jerusalem", "rejected", "pizza", "pickup", 0.0),
    ("12. Obsolete croissant", "croissant", 1, "Monday", "Jerusalem", "rejected", "croissant", "pickup", 0.0),
]
results = []
for label, item, quantity, day, city, status, sold, delivery, total in TEST_CASES:
    results.append(run_order(label, dict(item=item, quantity=quantity, day=day, city=city),
                             dict(status=status, item=sold, delivery=delivery, total=total)))
assert "only inside Jerusalem" in results[2]["message"]
assert "substituted rugelach" in results[3]["message"]
assert results[4]["suggestion_count"] == 2
assert results[6]["goods_subtotal"] == 180.0
assert results[6]["discounted_goods"] == 153.0
assert results[6]["delivery_fee"] == 20.0
assert results[7]["discount_amount"] == 0.0
assert results[8]["current_item"] == "rugelach"
assert [r["suggestion_count"] for r in results[3:6]] == [1, 2, 2]

## 10. Boundary, normalization, validation, and isolation checks
No accepted fixed-inventory order reaches exactly 150 NIS. Test the actual pricing node with a controlled state at that boundary; the menu, inventory, and compiled graph remain unchanged.

In [ ]:
boundary = calculate_total({"unit_price": 30.0, "quantity": 5, "day": "Thursday",
                            "delivery_fee": 20.0, "steps": []})
assert boundary["goods_subtotal"] == 150.0
assert boundary["discount_amount"] == 0.0
assert boundary["discounted_goods"] == 150.0
assert boundary["total"] == 170.0
pprint({"controlled_150_NIS_boundary": boundary})

base = {"item": "challah", "quantity": 2, "day": "Monday", "city": ""}
for field in REQUIRED_INPUTS:
    incomplete = {key: value for key, value in base.items() if key != field}
    run_order(f"Missing {field}", incomplete, {"status": "rejected", "total": 0.0})
for quantity in (-1, True, False, 1.0, "2", None, [], {}):
    run_order(f"Invalid quantity {quantity!r}", {**base, "quantity": quantity}, {"status": "rejected"})
for field, bad_values in (("item", (None, 7, [], "")),
                          ("day", (None, 7, [], "Funday")),
                          ("city", (7, [], False))):
    for value in bad_values:
        run_order(f"Invalid {field}: {value!r}", {**base, field: value}, {"status": "rejected"})
for city in (None, "   "):
    run_order(f"Pickup city {city!r}", {**base, "city": city},
              {"status": "accepted", "delivery": "pickup", "total": 36.0})
run_order("Normalized Friday", {"item": " CHALLAH ", "quantity": 10,
          "day": " fRiDaY ", "city": " jERUSALEM "},
          {"status": "accepted", "item": "challah", "day": "Friday", "city": "Jerusalem",
           "total": 200.0, "discount_amount": 0.0})
run_order("Discount after substitution", {"item": "sourdough", "quantity": 10,
          "day": "Monday", "city": ""}, {"item": "challah", "total": 153.0,
          "goods_subtotal": 180.0, "discount_amount": 27.0})

order = {"item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"}
first = graph.invoke(order)
second = graph.invoke(order)
assert first == second
assert first["steps"] is not second["steps"]
first["steps"].append("Caller-side mutation")
assert graph.invoke(order) == second
assert INVENTORY == {"challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4}
assert order == {"item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"}

# The exact requested public invocation works without config or credentials.
result = graph.invoke({
    "item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"
})
assert PUBLIC_KEYS <= result.keys() and result["total"] == 80.0
print("All required scenarios and additional checks passed.")

## 11. Summary
The story becomes a `TypedDict` state shared by nine small Python nodes. Validation rejects malformed orders before stock checks. Conditional edges after validation and every stock check enforce the circular sequence and two-alternative limit. Successful orders continue through fulfilment, pricing, and deterministic message templates; rejected orders finish immediately with a zero total and pickup default.

The diary records each executed node in order. Tests cover every conditional edge, both successful substitute positions, cancellation, delivery choices, discount and Friday rules, the exact threshold, invalid types and missing fields, normalization, and independent repeated calls.

API references: [LangGraph StateGraph](https://reference.langchain.com/python/langgraph/graph/state/StateGraph) and [draw_mermaid_png](https://reference.langchain.com/python/langchain-core/runnables/graph/Graph/draw_mermaid_png).

## Bonus Extension: Dynamic Inventory Management

The required `graph` above still checks the same fixed inventory on every order. This bonus adds a separate `inventory_graph`: accepted orders reduce stock for later orders in the same Colab runtime. Restarting the runtime resets stock. This is an educational example for sequential orders in one session, with no database persistence or concurrency support.

Upload this notebook to Colab and select **Runtime → Run all**. Colab already includes pandas for the inventory table. Internet access is needed for installation and the two graph diagrams; order processing runs locally.

### Bonus 1. Inventory and stock utilities

The live inventory starts as a copy of the fixed inventory. **Low stock** means one or two units; zero means **Out of stock**. The table shows current minus initial quantity, so restocks can make the difference positive.

In [ ]:
import pandas as pd

CURRENT_INVENTORY = INVENTORY.copy()
LOW_STOCK_THRESHOLD = 2


def reset_inventory():
    CURRENT_INVENTORY.clear()
    CURRENT_INVENTORY.update(INVENTORY)


def show_inventory():
    rows = []
    for product, initial in INVENTORY.items():
        current = CURRENT_INVENTORY[product]
        if current == 0:
            stock_status = "Out of stock"
        elif current <= LOW_STOCK_THRESHOLD:
            stock_status = "Low stock"
        else:
            stock_status = "In stock"
        rows.append({
            "Product": product,
            "Current quantity": current,
            "Initial quantity": initial,
            "Change from initial": current - initial,
            "Stock status": stock_status,
        })
    return pd.DataFrame(rows)


def restock_inventory(item: object, quantity: object) -> str:
    product = item.strip().lower() if isinstance(item, str) else ""
    if product not in MENU:
        raise ValueError("Choose a product from the current menu.")
    if type(quantity) is not int or quantity <= 0:
        raise ValueError("Restock quantity must be a positive integer.")
    before = CURRENT_INVENTORY[product]
    CURRENT_INVENTORY[product] += quantity
    return f"Restocked {product} by {quantity}: {before} → {CURRENT_INVENTORY[product]}."

### Bonus 2. Live stock and reservation

Each result carries a copy of the live inventory. For a sale, `inventory_before` and `inventory_after` are counts for the product actually sold. For a rejection, both stay `None` and `stock_updated` stays false. The stock router still checks the second substitute before deciding whether to cancel.

In [ ]:
class InventoryOrderState(OrderState, total=False):
    inventory_before: int | None
    inventory_after: int | None
    inventory_snapshot: dict[str, int]
    stock_updated: bool


def initialize_inventory_order(state: InventoryOrderState) -> dict:
    return {
        **initialize_order(state),
        "inventory_before": None,
        "inventory_after": None,
        "inventory_snapshot": CURRENT_INVENTORY.copy(),
        "stock_updated": False,
    }


def check_live_stock(state: InventoryOrderState) -> dict:
    product = state["current_item"]
    quantity = state["quantity"]
    available = CURRENT_INVENTORY[product]
    enough = available >= quantity
    attempt = state["suggestion_count"]
    label = f"substitute {attempt}" if attempt else "requested product"
    decision = "sufficient" if enough else "insufficient"
    return {
        "in_stock": enough,
        "steps": state["steps"] + [
            f"check_live_stock: {label} {product}; need {quantity}, "
            f"available {available}; {decision}."
        ],
    }


def reserve_inventory(state: InventoryOrderState) -> dict:
    if state["status"] != "accepted" or state["stock_updated"]:
        raise RuntimeError("Only an accepted order can reserve stock, once.")
    sold = state["current_item"]
    quantity = state["quantity"]
    before = CURRENT_INVENTORY[sold]
    if not 0 < quantity <= before:
        raise RuntimeError("Cannot reserve more than the available stock.")

    after = before - quantity
    CURRENT_INVENTORY[sold] = after
    unit = "boxes" if sold == "rugelach" else "units"
    return {
        "inventory_before": before,
        "inventory_after": after,
        "inventory_snapshot": CURRENT_INVENTORY.copy(),
        "stock_updated": True,
        "steps": state["steps"] + [
            f"reserve_inventory: reduced {sold} stock from {before} to {after} "
            f"after selling {quantity} {unit}."
        ],
    }

### Bonus 3. Compile and draw the separate inventory graph

Validation, substitution routing, delivery, pricing, and customer messages reuse the original functions. Only the accepted branch reaches the reservation node.

In [ ]:
inventory_builder = StateGraph(InventoryOrderState, input_schema=OrderInput)
inventory_builder.add_node("initialize_order", initialize_inventory_order)
for node in (validate_order, reject_invalid_order, check_live_stock, try_substitute,
             cancel_unavailable_order, determine_delivery, calculate_total,
             reserve_inventory, create_message):
    inventory_builder.add_node(node.__name__, node)

inventory_builder.add_edge(START, "initialize_order")
inventory_builder.add_edge("initialize_order", "validate_order")
inventory_builder.add_conditional_edges("validate_order", route_validation, {
    "valid": "check_live_stock", "invalid": "reject_invalid_order"
})
inventory_builder.add_conditional_edges("check_live_stock", route_stock, {
    "available": "determine_delivery", "substitute": "try_substitute",
    "cancel": "cancel_unavailable_order"
})
inventory_builder.add_edge("try_substitute", "check_live_stock")
inventory_builder.add_edge("reject_invalid_order", END)
inventory_builder.add_edge("cancel_unavailable_order", END)
inventory_builder.add_edge("determine_delivery", "calculate_total")
inventory_builder.add_edge("calculate_total", "reserve_inventory")
inventory_builder.add_edge("reserve_inventory", "create_message")
inventory_builder.add_edge("create_message", END)
inventory_graph = inventory_builder.compile()

In [ ]:
display(Image(inventory_graph.get_graph().draw_mermaid_png()))

### Bonus 4. Run five orders against the same inventory

The runner checks the complete inventory change on every call. A sale must reduce exactly the sold product by the order quantity, with one reservation step. A rejected order must leave every count unchanged.

In [ ]:
def run_inventory_order(label, order):
    before = CURRENT_INVENTORY.copy()
    result = inventory_graph.invoke(order)
    after = CURRENT_INVENTORY.copy()
    accepted = result["status"] == "accepted"
    sold = result["item"] if accepted else None
    reservations = [
        step for step in result["steps"] if step.startswith("reserve_inventory:")
    ]

    assert PUBLIC_KEYS <= result.keys()
    assert all(count >= 0 for count in after.values())
    assert result["inventory_snapshot"] == after
    assert result["inventory_snapshot"] is not CURRENT_INVENTORY
    assert result["stock_updated"] is accepted
    if accepted:
        expected = before.copy()
        expected[sold] -= order["quantity"]
        assert after == expected
        assert len(reservations) == 1
        assert result["inventory_before"] == before[sold]
        assert result["inventory_after"] == after[sold]
    else:
        assert result["status"] == "rejected"
        assert after == before and not reservations
        assert result["inventory_before"] is None and result["inventory_after"] is None
        assert result["total"] == 0.0 and result["delivery"] == "pickup"

    print()
    print(label)
    print("Order:", order)
    print("Inventory before:", before)
    print("Result:")
    pprint({key: value for key, value in result.items() if key != "steps"})
    print("Item sold:", sold or "None")
    print("Inventory after:", after)
    print("Steps:")
    for step in result["steps"]:
        print(f"  {step}")
    return result


def live_stock_checks(result):
    return [step for step in result["steps"] if step.startswith("check_live_stock:")]


reset_inventory()
assert CURRENT_INVENTORY == INVENTORY
display(show_inventory())

challah_order = {"item": "challah", "quantity": 4, "day": "Monday", "city": ""}
challah_first = run_inventory_order("Order 1: first challah sale", challah_order)
challah_second = run_inventory_order("Order 2: second challah sale", challah_order)
for sale, counts in ((challah_first, (10, 6)), (challah_second, (6, 2))):
    assert sale["status"] == "accepted" and sale["item"] == "challah"
    assert (sale["inventory_before"], sale["inventory_after"]) == counts

challah_third = run_inventory_order("Order 3: no candidate can fill the order", challah_order)
assert challah_third["status"] == "rejected"
assert live_stock_checks(challah_third) == [
    "check_live_stock: requested product challah; need 4, available 2; insufficient.",
    "check_live_stock: substitute 1 babka; need 4, available 0; insufficient.",
    "check_live_stock: substitute 2 rugelach; need 4, available 3; insufficient.",
]

babka_order = {"item": "babka", "quantity": 2, "day": "Tuesday", "city": "Jerusalem"}
babka_first = run_inventory_order("Order 4: substitute rugelach", babka_order)
assert babka_first["status"] == "accepted" and babka_first["item"] == "rugelach"
assert babka_first["suggestion_count"] == 1 and babka_first["total"] == 80.0
assert (babka_first["inventory_before"], babka_first["inventory_after"]) == (3, 1)
assert CURRENT_INVENTORY["babka"] == 0

babka_second = run_inventory_order("Order 5: use the updated rugelach stock", babka_order)
assert babka_second["status"] == "accepted" and babka_second["item"] == "sourdough"
assert babka_second["suggestion_count"] == 2 and babka_second["total"] == 70.0
assert (babka_second["inventory_before"], babka_second["inventory_after"]) == (4, 2)
assert live_stock_checks(babka_second) == [
    "check_live_stock: requested product babka; need 2, available 0; insufficient.",
    "check_live_stock: substitute 1 rugelach; need 2, available 1; insufficient.",
    "check_live_stock: substitute 2 sourdough; need 2, available 4; sufficient.",
]
assert CURRENT_INVENTORY == {"challah": 2, "babka": 0, "rugelach": 1, "sourdough": 2}
assert babka_first["inventory_snapshot"]["sourdough"] == 4
display(show_inventory())

### Bonus 5. Protect stock, restock, and reset

Check invalid and unfulfillable orders before restocking. Then compare the original graph's twelve full results with their earlier values while live stock differs from its initial counts.

In [ ]:
before_invalid = CURRENT_INVENTORY.copy()
invalid = run_inventory_order("Invalid order: zero quantity", {**challah_order, "quantity": 0})
assert invalid["status"] == "rejected" and not live_stock_checks(invalid)
assert CURRENT_INVENTORY == before_invalid

before_rejection = CURRENT_INVENTORY.copy()
rejected = run_inventory_order("Valid order: insufficient stock", {**babka_order, "quantity": 8})
assert rejected["status"] == "rejected" and len(live_stock_checks(rejected)) == 3
assert CURRENT_INVENTORY == before_rejection

before_restock = CURRENT_INVENTORY.copy()
print(restock_inventory(" Rugelach ", 5))
assert CURRENT_INVENTORY == {**before_restock, "rugelach": before_restock["rugelach"] + 5}
display(show_inventory())

after_restock = CURRENT_INVENTORY.copy()
bad_restocks = [("pizza", 1), (None, 1)] + [
    ("rugelach", quantity) for quantity in (0, -1, True, 2.0, "2", None, [])
]
for product, quantity in bad_restocks:
    try:
        restock_inventory(product, quantity)
    except ValueError as error:
        print(f"Rejected restock ({product!r}, {quantity!r}): {error}")
    else:
        raise AssertionError("An invalid restock was accepted.")
    assert CURRENT_INVENTORY == after_restock

for case, expected in zip(TEST_CASES, results):
    _, item, quantity, day, city, *_ = case
    assert graph.invoke(dict(item=item, quantity=quantity, day=day, city=city)) == expected
assert CURRENT_INVENTORY == after_restock

live_inventory_reference = CURRENT_INVENTORY
reset_inventory()
assert CURRENT_INVENTORY is live_inventory_reference
assert CURRENT_INVENTORY == INVENTORY == {
    "challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4
}
assert babka_second["inventory_snapshot"] == before_invalid
display(show_inventory())
print("All dynamic inventory bonus checks passed.")